# Public RAG Dataset Collector

This Colab notebook programmatically downloads the following public RAG evaluation datasets:

- BEIR
- Natural Questions (NQ)
- HotpotQA
- MuSiQue
- FEVER
- MultiHop-RAG
- RAGBench
- Open RAG Benchmark
- CRAG
- RGB

After **every dataset step**, the notebook prints:

- the completed dataset's on-disk size,
- the exact number of bytes, and
- remaining Colab disk space.

## Storage profiles

The default `colab_safe` profile downloads every dataset family while avoiding some exceptionally large optional components:

- a compact, diverse BEIR subset instead of all BEIR collections,
- the Natural Questions validation split instead of all splits,
- FEVER claims without the large Wikipedia-pages configuration,
- CRAG Tasks 1 and 2 without the larger Task 3 archive.

Change `PROFILE = "full"` in the configuration cell to request the larger components. The full profile can exceed the storage available in a standard Colab runtime. Select `size_only` to inspect the full profile's remotely reported compressed/repository sizes without downloading dataset contents. Exact extracted sizes are generally unavailable until download and conversion.

> Public download availability does not imply unrestricted commercial use. Check each dataset's current license and the licenses of upstream source corpora before redistribution or commercial use.

In [18]:
# from google.colab import drive
# drive.mount('/content/drive')

In [19]:
# Install download and dataset utilities.
# datasets<3 preserves compatibility with repositories that still use loading scripts.
%pip install -q "datasets>=2.19,<3" "huggingface_hub>=0.24" beir gdown tqdm pandas requests

## 1. Configuration

In [20]:
from pathlib import Path

# "colab_safe" downloads all ten dataset families with smaller/default components.
# "full" downloads the largest optional components and may exceed Colab storage.
# "size_only" reports remote sizes for the full selection without downloading data.
PROFILE = "colab_safe"  #@param ["colab_safe", "full", "size_only"]

ROOT = Path("/content/rag_datasets")
FORCE_REDOWNLOAD = False  #@param {type:"boolean"}

# BEIR: compact profile is deliberately diverse and avoids duplicating NQ/HotpotQA.
BEIR_COMPACT = [
    "arguana",
    "fiqa",
    "nfcorpus",
    "scifact",
    "trec-covid",
]

BEIR_ALL = [
    "arguana",
    "climate-fever",
    "cqadupstack",
    "dbpedia-entity",
    "fever",
    "fiqa",
    "hotpotqa",
    "msmarco",
    "nfcorpus",
    "nq",
    "quora",
    "robust04",
    "scidocs",
    "scifact",
    "signal1m",
    "trec-covid",
    "trec-news",
    "webis-touche2020",
]

SIZE_ONLY = PROFILE == "size_only"
FULL_SELECTION = PROFILE in {"full", "size_only"}

BEIR_DATASETS = BEIR_ALL if FULL_SELECTION else BEIR_COMPACT
NQ_SPLIT = None if FULL_SELECTION else "validation"
HOTPOT_CONFIG = "distractor"
FEVER_INCLUDE_WIKI = FULL_SELECTION
CRAG_INCLUDE_TASK3 = FULL_SELECTION

if not SIZE_ONLY:
    ROOT.mkdir(parents=True, exist_ok=True)

print(f"Profile: {PROFILE}")
print(f"Destination: {ROOT}")
print(f"BEIR collections: {len(BEIR_DATASETS)}")
print(f"Natural Questions: {'all splits' if NQ_SPLIT is None else NQ_SPLIT}")
print(f"FEVER Wikipedia pages: {FEVER_INCLUDE_WIKI}")
print(f"CRAG Task 3: {CRAG_INCLUDE_TASK3}")

Profile: colab_safe
Destination: /content/rag_datasets
BEIR collections: 5
Natural Questions: validation
FEVER Wikipedia pages: False
CRAG Task 3: False


## 2. Shared download and size-reporting helpers

In [21]:
import bz2
import re
import shutil
import subprocess
import tarfile
import time
from pathlib import Path

import pandas as pd
import requests
from huggingface_hub import HfApi
from IPython.display import display
from tqdm.auto import tqdm

DOWNLOAD_RESULTS = {}

def human_bytes(num_bytes) -> str:
    if num_bytes is None:
        return "unknown"
    value = float(num_bytes)
    units = ["B", "KiB", "MiB", "GiB", "TiB"]
    for unit in units:
        if value < 1024 or unit == units[-1]:
            return f"{value:,.2f} {unit}"
        value /= 1024
    return f"{num_bytes:,} B"

def remote_content_length(url: str):
    """Return a remote file's byte length without consuming its body."""
    headers = {"Accept-Encoding": "identity"}
    try:
        with requests.head(
            url, headers=headers, allow_redirects=True, timeout=60
        ) as response:
            response.raise_for_status()
            content_type = response.headers.get("content-type", "").lower()
            length = response.headers.get("content-length")
            if length and "text/html" not in content_type:
                return int(length)
    except requests.RequestException:
        pass

    range_headers = {**headers, "Range": "bytes=0-0"}
    try:
        with requests.get(
            url, headers=range_headers, stream=True, allow_redirects=True, timeout=60
        ) as response:
            response.raise_for_status()
            content_range = response.headers.get("content-range", "")
            match = re.search(r"/(\d+)$", content_range)
            if match:
                return int(match.group(1))
            content_type = response.headers.get("content-type", "").lower()
            length = response.headers.get("content-length")
            if length and "text/html" not in content_type:
                return int(length)
    except requests.RequestException:
        pass
    return None

def hf_dataset_viewer_size(repo_id: str, configs=None):
    """Return original-file bytes reported by the HF Dataset Viewer."""
    response = requests.get(
        "https://datasets-server.huggingface.co/size",
        params={"dataset": repo_id},
        timeout=60,
    )
    response.raise_for_status()
    size_data = response.json()["size"]

    if configs is None:
        dataset = size_data["dataset"]
        return dataset.get("num_bytes_original_files"), dataset.get("num_rows")

    wanted = set(configs)
    matched = [item for item in size_data.get("configs", []) if item.get("config") in wanted]
    found = {item.get("config") for item in matched}
    if found != wanted:
        raise RuntimeError(f"Missing size metadata for configs: {sorted(wanted - found)}")
    total_bytes = sum(item.get("num_bytes_original_files") or 0 for item in matched)
    total_rows = sum(item.get("num_rows") or 0 for item in matched)
    return total_bytes, total_rows

def hf_dataset_card_size(repo_id: str, configs):
    """Return download bytes recorded in Hugging Face dataset-card metadata."""
    response = requests.get(
        f"https://huggingface.co/api/datasets/{repo_id}", timeout=60
    )
    response.raise_for_status()
    card_data = response.json().get("cardData") or {}
    dataset_info = card_data.get("dataset_info") or []
    if isinstance(dataset_info, dict):
        dataset_info = [dataset_info]

    wanted = set(configs)
    matched = [item for item in dataset_info if item.get("config_name") in wanted]
    found = {item.get("config_name") for item in matched}
    if found != wanted:
        raise RuntimeError(f"Missing dataset-card sizes for configs: {sorted(wanted - found)}")
    if any(item.get("download_size") is None for item in matched):
        raise RuntimeError("Dataset-card download_size metadata is incomplete")

    total_bytes = sum(item["download_size"] for item in matched)
    total_rows = sum(
        split.get("num_examples", 0)
        for item in matched
        for split in item.get("splits", [])
    )
    return total_bytes, total_rows

def hf_repository_size(repo_id: str):
    """Sum file sizes in a Hugging Face dataset repository."""
    info = HfApi().dataset_info(repo_id, files_metadata=True)
    missing = [item.rfilename for item in info.siblings if item.size is None]
    if missing:
        raise RuntimeError(f"Size metadata missing for {len(missing)} repository files")
    return sum(item.size for item in info.siblings), len(info.siblings)

def github_repository_blob_size(repo: str, ref: str):
    """Sum Git blob sizes from a repository tree without cloning it."""
    response = requests.get(
        f"https://api.github.com/repos/{repo}/git/trees/{ref}",
        params={"recursive": 1},
        timeout=60,
    )
    response.raise_for_status()
    payload = response.json()
    if payload.get("truncated"):
        raise RuntimeError("GitHub returned a truncated repository tree")
    blobs = [item for item in payload.get("tree", []) if item.get("type") == "blob"]
    return sum(item.get("size", 0) for item in blobs), len(blobs)

def size_metadata(size_bytes, source, kind, note="", rows=None, complete=True):
    return {
        "size_bytes": size_bytes,
        "source": source,
        "size_kind": kind,
        "note": note,
        "rows": rows,
        "complete": complete,
    }

def path_size_bytes(path: Path) -> int:
    """Return the physical size of regular files under path."""
    path = Path(path)
    if not path.exists():
        return 0
    if path.is_file():
        return path.stat().st_size

    total = 0
    for item in path.rglob("*"):
        try:
            if item.is_file() and not item.is_symlink():
                total += item.stat().st_size
        except FileNotFoundError:
            # A temporary file may disappear while a library finalizes a download.
            pass
    return total

def remaining_disk_bytes(path: Path = Path("/content")) -> int:
    return shutil.disk_usage(path).free

def nonempty(path: Path) -> bool:
    path = Path(path)
    return path.exists() and (path.is_file() or any(path.iterdir()))

def prepare_destination(path: Path) -> None:
    path = Path(path)
    if FORCE_REDOWNLOAD and path.exists():
        if path.is_dir():
            shutil.rmtree(path)
        else:
            path.unlink()
    path.mkdir(parents=True, exist_ok=True)

def run_command(command, cwd=None) -> None:
    printable = " ".join(map(str, command))
    print(f"$ {printable}")
    subprocess.run(
        list(map(str, command)),
        cwd=str(cwd) if cwd else None,
        check=True,
    )

def download_file(url: str, destination: Path) -> Path:
    """Stream a URL to disk with a progress bar and atomic rename."""
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)

    if destination.exists() and destination.stat().st_size > 0 and not FORCE_REDOWNLOAD:
        print(f"Using existing file: {destination.name}")
        return destination

    temporary = destination.with_suffix(destination.suffix + ".part")
    if temporary.exists():
        temporary.unlink()

    with requests.get(url, stream=True, timeout=(30, 300)) as response:
        response.raise_for_status()
        total = int(response.headers.get("content-length", 0))
        with temporary.open("wb") as output, tqdm(
            total=total or None,
            unit="B",
            unit_scale=True,
            desc=destination.name,
        ) as progress:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    output.write(chunk)
                    progress.update(len(chunk))

    temporary.replace(destination)
    return destination

def safe_extract_tar(archive: Path, destination: Path) -> None:
    """Extract a tar archive while rejecting path traversal."""
    archive = Path(archive)
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()

    with tarfile.open(archive, "r:*") as tar:
        for member in tar.getmembers():
            target = (destination / member.name).resolve()
            if target != root and root not in target.parents:
                raise RuntimeError(f"Unsafe path in archive: {member.name}")
        tar.extractall(destination)

def report_dataset(
    name: str,
    path: Path,
    started_at: float,
    status: str = "complete",
    error: str = "",
):
    size = path_size_bytes(path)
    elapsed = time.time() - started_at
    free = remaining_disk_bytes()

    DOWNLOAD_RESULTS[name] = {
        "dataset": name,
        "mode": "download",
        "status": status,
        "path": str(path),
        "size_bytes": size,
        "size": human_bytes(size),
        "elapsed_seconds": round(elapsed, 1),
        "size_kind": "local on-disk footprint",
        "rows": None,
        "note": "",
        "error": error,
    }

    if status == "complete":
        print(f"\n✅ {name} download step complete")
        print(f"   On-disk size: {human_bytes(size)} ({size:,} bytes)")
        print(f"   Location: {path}")
        print(f"   Remaining Colab disk: {human_bytes(free)}")
    else:
        print(f"\n❌ {name} failed: {error}")
        print(f"   Partial on-disk size: {human_bytes(size)} ({size:,} bytes)")
        print(f"   Remaining Colab disk: {human_bytes(free)}")

def report_size_only(name: str, metadata: dict, elapsed: float):
    size = metadata.get("size_bytes")
    status = (
        "size_available"
        if size is not None and metadata.get("complete", True)
        else "size_partial" if size is not None else "size_unavailable"
    )
    DOWNLOAD_RESULTS[name] = {
        "dataset": name,
        "mode": "size_only",
        "status": status,
        "path": metadata.get("source", ""),
        "size_bytes": size,
        "size": human_bytes(size),
        "elapsed_seconds": round(elapsed, 1),
        "size_kind": metadata.get("size_kind", "remote metadata"),
        "rows": metadata.get("rows"),
        "note": metadata.get("note", ""),
        "error": "",
    }
    icon = "✅" if status == "size_available" else "⚠️"
    print(f"\n{icon} {name} size inspection complete ({status})")
    print(f"   Reported size: {human_bytes(size)}")
    print(f"   Size type: {metadata.get('size_kind', 'remote metadata')}")
    if metadata.get("rows") is not None:
        print(f"   Rows: {metadata['rows']:,}")
    if metadata.get("note"):
        print(f"   Note: {metadata['note']}")

def execute_step(name: str, path: Path, downloader, size_reporter=None):
    print("=" * 80)
    print(f"{'Inspecting full size' if SIZE_ONLY else 'Downloading'}: {name}")
    print("=" * 80)
    started = time.time()
    try:
        if SIZE_ONLY:
            if size_reporter is None:
                raise RuntimeError("No remote size reporter is configured")
            report_size_only(name, size_reporter(), time.time() - started)
        else:
            downloader()
            report_dataset(name, path, started, status="complete")
    except Exception as exc:
        if SIZE_ONLY:
            report_size_only(
                name,
                size_metadata(None, "", "unavailable", note=repr(exc)),
                time.time() - started,
            )
        else:
            report_dataset(name, path, started, status="failed", error=repr(exc))
            print("This notebook continues so the remaining datasets can still be collected.")

if SIZE_ONLY:
    print("Size-only mode: dataset contents will not be downloaded.")
else:
    print(f"Initial free disk: {human_bytes(remaining_disk_bytes())}")

Initial free disk: 205.83 GiB


## 3. BEIR

Uses the official BEIR download utility and public UKP dataset archives.

The safe profile downloads five heterogeneous collections; the full profile requests all collections listed in the configuration.

In [22]:
from beir import util as beir_util

beir_path = ROOT / "BEIR"
beir_base_url = (
    "https://public.ukp.informatik.tu-darmstadt.de/"
    "thakur/BEIR/datasets"
)

def size_beir():
    total = 0
    unavailable = []
    for dataset_name in BEIR_DATASETS:
        url = f"{beir_base_url}/{dataset_name}.zip"
        size = remote_content_length(url)
        print(f"   {dataset_name}: {human_bytes(size)}")
        if size is None:
            unavailable.append(dataset_name)
        else:
            total += size
    return size_metadata(
        total,
        beir_base_url,
        "compressed archive download size" if not unavailable else "known compressed archive subtotal",
        note=(
            "Extracted size will be larger. "
            + (f"Missing metadata for: {', '.join(unavailable)}" if unavailable else "")
        ).strip(),
        complete=not unavailable,
    )

def download_beir():
    prepare_destination(beir_path)

    for dataset_name in BEIR_DATASETS:
        expected = beir_path / dataset_name
        if nonempty(expected) and not FORCE_REDOWNLOAD:
            print(f"Using existing BEIR collection: {dataset_name}")
        else:
            url = f"{beir_base_url}/{dataset_name}.zip"
            extracted_path = Path(
                beir_util.download_and_unzip(url, str(beir_path))
            )
            print(f"Downloaded {dataset_name} -> {extracted_path}")

        collection_size = path_size_bytes(expected)
        print(
            f"   {dataset_name}: "
            f"{human_bytes(collection_size)} ({collection_size:,} bytes)"
        )

execute_step("BEIR", beir_path, download_beir, size_beir)

Downloading: BEIR


/content/rag_datasets/BEIR/arguana.zip:   0%|          | 0.00/3.60M [00:00<?, ?iB/s]

Downloaded arguana -> /content/rag_datasets/BEIR/arguana
   arguana: 11.19 MiB (11,731,700 bytes)


/content/rag_datasets/BEIR/fiqa.zip:   0%|          | 0.00/17.1M [00:00<?, ?iB/s]

Downloaded fiqa -> /content/rag_datasets/BEIR/fiqa
   fiqa: 46.64 MiB (48,909,169 bytes)


...

Downloaded nfcorpus -> /content/rag_datasets/BEIR/nfcorpus
   nfcorpus: 9.25 MiB (9,703,263 bytes)


/content/rag_datasets/BEIR/scifact.zip:   0%|          | 0.00/2.69M [00:00<?, ?iB/s]

Downloaded scifact -> /content/rag_datasets/BEIR/scifact
   scifact: 7.95 MiB (8,336,188 bytes)


/content/rag_datasets/BEIR/trec-covid.zip:   0%|          | 0.00/70.5M [00:00<?, ?iB/s]

Downloaded trec-covid -> /content/rag_datasets/BEIR/trec-covid
   trec-covid: 212.07 MiB (222,367,448 bytes)

✅ BEIR download step complete
   On-disk size: 383.29 MiB (401,910,643 bytes)
   Location: /content/rag_datasets/BEIR
   Remaining Colab disk: 205.45 GiB


## 4. Natural Questions (NQ)

Downloads from the official Hugging Face organization.

- `colab_safe`: validation split
- `full`: every available split

The full original Natural Questions release is exceptionally large. This step stores the processed Hugging Face dataset under the destination directory and removes its temporary conversion cache afterward.

In [23]:
from datasets import load_dataset

nq_path = ROOT / "Natural_Questions"

def size_natural_questions():
    size, rows = hf_dataset_card_size(
        "google-research-datasets/natural_questions", configs=["default"]
    )
    return size_metadata(
        size,
        "google-research-datasets/natural_questions",
        "download size reported by Hugging Face dataset card",
        note="Full dataset; prepared Arrow/on-disk size may be larger.",
        rows=rows,
    )

def download_natural_questions():
    prepare_destination(nq_path)
    output = nq_path / "dataset"
    cache = nq_path / "_temporary_hf_cache"

    if nonempty(output) and not FORCE_REDOWNLOAD:
        print("Using existing Natural Questions dataset.")
        return

    if cache.exists():
        shutil.rmtree(cache)

    kwargs = {
        "path": "google-research-datasets/natural_questions",
        "cache_dir": str(cache),
    }
    if NQ_SPLIT is not None:
        kwargs["split"] = NQ_SPLIT

    dataset = load_dataset(**kwargs)
    dataset.save_to_disk(str(output))

    del dataset
    if cache.exists():
        shutil.rmtree(cache)

execute_step(
    "Natural Questions (NQ)", nq_path, download_natural_questions, size_natural_questions
)

Downloading: Natural Questions (NQ)


Resolving data files:   0%|          | 0/287 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/287 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/307373 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7830 [00:00<?, ? examples/s]

Saving the dataset (0/7 shards):   0%|          | 0/7830 [00:00<?, ? examples/s]


✅ Natural Questions (NQ) download step complete
   On-disk size: 3.21 GiB (3,451,427,851 bytes)
   Location: /content/rag_datasets/Natural_Questions
   Remaining Colab disk: 202.24 GiB


## 5. HotpotQA

Downloads the `distractor` configuration from the official Hugging Face dataset repository. It includes supporting-fact annotations and is practical for retrieval-plus-reasoning evaluation.

In [24]:
hotpot_path = ROOT / "HotpotQA"

def size_hotpotqa():
    size, rows = hf_dataset_viewer_size(
        "hotpotqa/hotpot_qa", configs=[HOTPOT_CONFIG]
    )
    return size_metadata(
        size,
        f"hotpotqa/hotpot_qa::{HOTPOT_CONFIG}",
        "original-file download size reported by Hugging Face",
        note="Prepared Arrow/on-disk size may be larger.",
        rows=rows,
    )

def download_hotpotqa():
    prepare_destination(hotpot_path)
    output = hotpot_path / "dataset"
    cache = hotpot_path / "_temporary_hf_cache"

    if nonempty(output) and not FORCE_REDOWNLOAD:
        print("Using existing HotpotQA dataset.")
        return

    if cache.exists():
        shutil.rmtree(cache)

    dataset = load_dataset(
        "hotpotqa/hotpot_qa",
        HOTPOT_CONFIG,
        cache_dir=str(cache),
    )
    dataset.save_to_disk(str(output))

    del dataset
    if cache.exists():
        shutil.rmtree(cache)

execute_step("HotpotQA", hotpot_path, download_hotpotqa, size_hotpotqa)

Downloading: HotpotQA


Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7405 [00:00<?, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/90447 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/7405 [00:00<?, ? examples/s]


✅ HotpotQA download step complete
   On-disk size: 571.04 MiB (598,782,259 bytes)
   Location: /content/rag_datasets/HotpotQA
   Remaining Colab disk: 201.68 GiB


## 6. MuSiQue

Clones the official repository and runs its official `download_data.sh` script. Downloaded files remain under the repository's `data/` directory.

In [25]:
musique_path = ROOT / "MuSiQue"
musique_archive_url = (
    "https://drive.usercontent.google.com/download"
    "?id=1tGdADlNjWFaHLeZZGShh2IRcpO6Lv24h&export=download&confirm=t"
)

def size_musique():
    archive_size = remote_content_length(musique_archive_url)
    repository_size, repository_files = github_repository_blob_size(
        "StonyBrookNLP/musique", "main"
    )
    print(f"   Git repository files: {human_bytes(repository_size)}")
    print(f"   musique_v1.0.zip: {human_bytes(archive_size)}")
    total = repository_size + archive_size if archive_size is not None else None
    return size_metadata(
        total,
        "StonyBrookNLP/musique plus Google Drive archive",
        "repository blobs plus compressed archive download size",
        note=(
            f"Repository contains {repository_files} files. Extracted size will be larger."
            if archive_size is not None
            else "Google Drive did not expose the archive size; exact total is unavailable."
        ),
    )

def download_musique():
    if FORCE_REDOWNLOAD and musique_path.exists():
        shutil.rmtree(musique_path)

    if not (musique_path / ".git").exists():
        run_command([
            "git", "clone", "--depth", "1",
            "https://github.com/StonyBrookNLP/musique.git",
            musique_path,
        ])

    data_path = musique_path / "data"
    if nonempty(data_path) and not FORCE_REDOWNLOAD:
        print("Using existing MuSiQue data directory.")
        return

    run_command(["bash", "download_data.sh"], cwd=musique_path)

execute_step("MuSiQue", musique_path, download_musique, size_musique)

Downloading: MuSiQue
$ git clone --depth 1 https://github.com/StonyBrookNLP/musique.git /content/rag_datasets/MuSiQue
$ bash download_data.sh

✅ MuSiQue download step complete
   On-disk size: 857.38 MiB (899,025,356 bytes)
   Location: /content/rag_datasets/MuSiQue
   Remaining Colab disk: 200.84 GiB


## 7. FEVER

Downloads FEVER through its Hugging Face loader.

- Both profiles download the `v1.0` claims/evidence configuration.
- The `full` profile additionally downloads `wiki_pages`, which requires substantially more disk space.

In [26]:
fever_path = ROOT / "FEVER"

def size_fever():
    configs = ["v1.0", "wiki_pages"]
    size, rows = hf_dataset_card_size("fever/fever", configs=configs)
    return size_metadata(
        size,
        "fever/fever::v1.0+wiki_pages",
        "download size reported by Hugging Face dataset card",
        note="Full selection including wiki_pages; prepared Arrow size may be larger.",
        rows=rows,
    )

def save_hf_configuration(repo_id: str, config: str, output: Path):
    cache = output.parent / f"_cache_{config.replace('.', '_')}"
    if nonempty(output) and not FORCE_REDOWNLOAD:
        print(f"Using existing configuration: {config}")
        return

    if cache.exists():
        shutil.rmtree(cache)

    dataset = load_dataset(
        repo_id,
        config,
        cache_dir=str(cache),
        trust_remote_code=True,
    )
    dataset.save_to_disk(str(output))
    del dataset

    if cache.exists():
        shutil.rmtree(cache)

def download_fever():
    prepare_destination(fever_path)

    save_hf_configuration(
        "fever/fever",
        "v1.0",
        fever_path / "v1_0",
    )

    if FEVER_INCLUDE_WIKI:
        save_hf_configuration(
            "fever/fever",
            "wiki_pages",
            fever_path / "wiki_pages",
        )
    else:
        print("Skipping FEVER wiki_pages in colab_safe profile.")

execute_step("FEVER", fever_path, download_fever, size_fever)

Downloading: FEVER


Generating train split:   0%|          | 0/311431 [00:00<?, ? examples/s]

Generating labelled_dev split:   0%|          | 0/37566 [00:00<?, ? examples/s]

Generating unlabelled_dev split:   0%|          | 0/19998 [00:00<?, ? examples/s]

Generating unlabelled_test split:   0%|          | 0/19998 [00:00<?, ? examples/s]

...

Saving the dataset (0/1 shards):   0%|          | 0/19998 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/19998 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/18999 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/18567 [00:00<?, ? examples/s]

Skipping FEVER wiki_pages in colab_safe profile.

✅ FEVER download step complete
   On-disk size: 38.22 MiB (40,077,008 bytes)
   Location: /content/rag_datasets/FEVER
   Remaining Colab disk: 200.80 GiB


## 8. MultiHop-RAG

Downloads a local snapshot of the public Hugging Face dataset repository.

In [27]:
from huggingface_hub import snapshot_download

multihop_rag_path = ROOT / "MultiHop-RAG"

def size_multihop_rag():
    size, files = hf_repository_size("yixuantt/MultiHopRAG")
    return size_metadata(
        size,
        "yixuantt/MultiHopRAG",
        "Hugging Face repository file size",
        note=f"Sum of {files} repository files.",
    )

def download_multihop_rag():
    if FORCE_REDOWNLOAD and multihop_rag_path.exists():
        shutil.rmtree(multihop_rag_path)

    snapshot_download(
        repo_id="yixuantt/MultiHopRAG",
        repo_type="dataset",
        local_dir=str(multihop_rag_path),
        local_dir_use_symlinks=False,
    )

execute_step(
    "MultiHop-RAG", multihop_rag_path, download_multihop_rag, size_multihop_rag
)

Downloading: MultiHop-RAG


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]


✅ MultiHop-RAG download step complete
   On-disk size: 11.41 MiB (11,961,535 bytes)
   Location: /content/rag_datasets/MultiHop-RAG
   Remaining Colab disk: 200.79 GiB


## 9. RAGBench

Downloads the complete public Hugging Face repository snapshot, including its available domain configurations.

In [28]:
ragbench_path = ROOT / "RAGBench"

def size_ragbench():
    size, files = hf_repository_size("galileo-ai/ragbench")
    return size_metadata(
        size,
        "galileo-ai/ragbench",
        "Hugging Face repository file size",
        note=f"Sum of {files} repository files.",
    )

def download_ragbench():
    if FORCE_REDOWNLOAD and ragbench_path.exists():
        shutil.rmtree(ragbench_path)

    snapshot_download(
        repo_id="galileo-ai/ragbench",
        repo_type="dataset",
        local_dir=str(ragbench_path),
        local_dir_use_symlinks=False,
    )

execute_step("RAGBench", ragbench_path, download_ragbench, size_ragbench)

Downloading: RAGBench


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 38 files:   0%|          | 0/38 [00:00<?, ?it/s]


✅ RAGBench download step complete
   On-disk size: 430.39 MiB (451,295,130 bytes)
   Location: /content/rag_datasets/RAGBench
   Remaining Colab disk: 200.37 GiB


## 10. Open RAG Benchmark

Downloads the raw public repository snapshot. A snapshot download is used because the benchmark contains source-oriented files useful for PDF and multimodal ingestion tests.

In [29]:
open_rag_path = ROOT / "Open_RAG_Benchmark"

def size_open_rag_benchmark():
    size, files = hf_repository_size("vectara/open_ragbench")
    return size_metadata(
        size,
        "vectara/open_ragbench",
        "Hugging Face repository file size",
        note=f"Sum of {files} repository files.",
    )

def download_open_rag_benchmark():
    if FORCE_REDOWNLOAD and open_rag_path.exists():
        shutil.rmtree(open_rag_path)

    snapshot_download(
        repo_id="vectara/open_ragbench",
        repo_type="dataset",
        local_dir=str(open_rag_path),
        local_dir_use_symlinks=False,
    )

execute_step(
    "Open RAG Benchmark",
    open_rag_path,
    download_open_rag_benchmark,
    size_open_rag_benchmark,
)

Downloading: Open RAG Benchmark


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1006 files:   0%|          | 0/1006 [00:00<?, ?it/s]


✅ Open RAG Benchmark download step complete
   On-disk size: 709.22 MiB (743,676,033 bytes)
   Location: /content/rag_datasets/Open_RAG_Benchmark
   Remaining Colab disk: 199.67 GiB


## 11. CRAG

Downloads official development data directly from the CRAG repository.

- Both profiles download and decompress Tasks 1 and 2.
- The `full` profile also downloads the four Task 3 archive parts, joins them, and safely extracts the resulting archive.

In [30]:
crag_path = ROOT / "CRAG"
crag_base_url = (
    "https://github.com/facebookresearch/CRAG/raw/"
    "refs/heads/main/data"
)

def size_crag():
    filenames = ["crag_task_1_and_2_dev_v4.jsonl.bz2"] + [
        f"crag_task_3_dev_v4.tar.bz2.part{part_number}"
        for part_number in range(1, 5)
    ]
    total = 0
    unavailable = []
    for filename in filenames:
        size = remote_content_length(f"{crag_base_url}/{filename}")
        print(f"   {filename}: {human_bytes(size)}")
        if size is None:
            unavailable.append(filename)
        else:
            total += size
    return size_metadata(
        None if unavailable else total,
        crag_base_url,
        "compressed archive download size",
        note=(
            "Full Tasks 1-3 selection; extracted size will be larger. "
            + (f"Missing metadata for: {', '.join(unavailable)}" if unavailable else "")
        ).strip(),
    )

def download_crag():
    prepare_destination(crag_path)

    task12_output = crag_path / "crag_task_1_and_2_dev_v4.jsonl"
    task12_archive = crag_path / "crag_task_1_and_2_dev_v4.jsonl.bz2"

    if not nonempty(task12_output) or FORCE_REDOWNLOAD:
        download_file(
            f"{crag_base_url}/crag_task_1_and_2_dev_v4.jsonl.bz2",
            task12_archive,
        )
        print("Decompressing CRAG Tasks 1 and 2...")
        with bz2.open(task12_archive, "rb") as source, task12_output.open("wb") as target:
            shutil.copyfileobj(source, target)
        task12_archive.unlink(missing_ok=True)
    else:
        print("Using existing CRAG Tasks 1 and 2 data.")

    if not CRAG_INCLUDE_TASK3:
        print("Skipping CRAG Task 3 in colab_safe profile.")
        return

    task3_directory = crag_path / "task3"
    if nonempty(task3_directory) and not FORCE_REDOWNLOAD:
        print("Using existing CRAG Task 3 data.")
        return

    part_paths = []
    for part_number in range(1, 5):
        part_name = f"crag_task_3_dev_v4.tar.bz2.part{part_number}"
        part_path = crag_path / part_name
        download_file(f"{crag_base_url}/{part_name}", part_path)
        part_paths.append(part_path)

    joined_archive = crag_path / "crag_task_3_dev_v4.tar.bz2"
    print("Joining CRAG Task 3 archive parts...")
    with joined_archive.open("wb") as output:
        for part_path in part_paths:
            with part_path.open("rb") as part:
                shutil.copyfileobj(part, output)

    print("Extracting CRAG Task 3...")
    safe_extract_tar(joined_archive, task3_directory)

    for part_path in part_paths:
        part_path.unlink(missing_ok=True)
    joined_archive.unlink(missing_ok=True)

execute_step("CRAG", crag_path, download_crag, size_crag)

Downloading: CRAG


crag_task_1_and_2_dev_v4.jsonl.bz2:   0%|          | 0.00/739M [00:00<?, ?B/s]

Decompressing CRAG Tasks 1 and 2...
Skipping CRAG Task 3 in colab_safe profile.

✅ CRAG download step complete
   On-disk size: 4.81 GiB (5,164,388,176 bytes)
   Location: /content/rag_datasets/CRAG
   Remaining Colab disk: 194.86 GiB


## 12. RGB

Clones the official repository, whose `data/` directory contains the English and Chinese robustness benchmark files.

In [31]:
rgb_path = ROOT / "RGB"

def size_rgb():
    size, files = github_repository_blob_size("chen700564/RGB", "master")
    return size_metadata(
        size,
        "chen700564/RGB@master",
        "Git repository blob size",
        note=f"Sum of {files} tracked files; excludes .git clone overhead.",
    )

def download_rgb():
    if FORCE_REDOWNLOAD and rgb_path.exists():
        shutil.rmtree(rgb_path)

    if not (rgb_path / ".git").exists():
        run_command([
            "git", "clone", "--depth", "1", "--branch", "master",
            "https://github.com/chen700564/RGB.git",
            rgb_path,
        ])
    else:
        print("Using existing RGB repository.")

execute_step("RGB", rgb_path, download_rgb, size_rgb)

Downloading: RGB
$ git clone --depth 1 --branch master https://github.com/chen700564/RGB.git /content/rag_datasets/RGB

✅ RGB download step complete
   On-disk size: 43.06 MiB (45,151,039 bytes)
   Location: /content/rag_datasets/RGB
   Remaining Colab disk: 194.82 GiB


## 13. Download summary and machine-readable manifest

In [32]:
summary = pd.DataFrame(DOWNLOAD_RESULTS.values())

if summary.empty:
    print("No download steps have been run in this session.")
else:
    ordered_columns = [
        "dataset",
        "mode",
        "status",
        "size",
        "size_bytes",
        "size_kind",
        "rows",
        "elapsed_seconds",
        "path",
        "note",
        "error",
    ]
    summary = summary[ordered_columns].sort_values("dataset").reset_index(drop=True)
    display(summary)

    known_sizes = pd.to_numeric(summary["size_bytes"], errors="coerce").dropna()
    total_bytes = int(known_sizes.sum())

    manifest_path = (
        Path("/content/rag_dataset_size_manifest.csv")
        if SIZE_ONLY
        else ROOT / "download_manifest.csv"
    )
    summary.to_csv(manifest_path, index=False)

    print(
        ("Total remotely reported size: " if SIZE_ONLY else "Total completed dataset footprint: ")
        + f"{human_bytes(total_bytes)} ({total_bytes:,} bytes)"
    )
    if SIZE_ONLY and (summary["status"] != "size_available").any():
        print("Warning: total uses known subtotals and excludes unavailable file sizes.")
    print(f"Manifest written to: {manifest_path}")
    if not SIZE_ONLY:
        print(f"Remaining Colab disk: {human_bytes(remaining_disk_bytes())}")

,dataset,mode,status,size,size_bytes,size_kind,rows,elapsed_seconds,path,note,error
0,BEIR,download,complete,383.29 MiB,401910643,local on-disk footprint,None,41.6,/content/rag_datasets/BEIR,,
1,CRAG,download,complete,4.81 GiB,5164388176,local on-disk footprint,None,192.0,/content/rag_datasets/CRAG,,
2,FEVER,download,complete,38.22 MiB,40077008,local on-disk footprint,None,15.2,/content/rag_datasets/FEVER,,
3,HotpotQA,download,complete,571.04 MiB,598782259,local on-disk footprint,None,35.5,/content/rag_datasets/HotpotQA,,
4,MuSiQue,download,complete,857.38 MiB,899025356,local on-disk footprint,None,15.3,/content/rag_datasets/MuSiQue,,
5,MultiHop-RAG,download,complete,11.41 MiB,11961535,local on-disk footprint,None,0.7,/content/rag_datasets/MultiHop-RAG,,
6,Natural Questions (NQ),download,complete,3.21 GiB,3451427851,local on-disk footprint,None,2583.3,/content/rag_datasets/Natural_Questions,,
7,Open RAG Benchmark,download,complete,709.22 MiB,743676033,local on-disk footprint,None,26.1,/content/rag_datasets/Open_RAG_Benchmark,,
8,RAGBench,download,complete,430.39 MiB,451295130,local on-disk footprint,None,10.4,/content/rag_datasets/RAGBench,,
9,RGB,download,complete,43.06 MiB,45151039,local on-disk footprint,None,1.2,/content/rag_datasets/RGB,,


Total completed dataset footprint: 11.00 GiB (11,807,695,030 bytes)
Manifest written to: /content/rag_datasets/download_manifest.csv
Remaining Colab disk: 194.82 GiB


## 14. Create portable subsets (up to 5,000 rows each)

This cell creates a reproducible sample for every Hugging Face split and supported tabular data file found under `ROOT`. Outputs are newline-delimited JSON under `/content/rag_dataset_subsets`, with their source-relative layout preserved. Unsupported or non-tabular files are left out and reported.

In [33]:
import itertools
import json
import re

from datasets import DatasetDict, load_from_disk

MAX_ROWS_PER_ITEM = 5_000  #@param {type:"integer"}
SUBSET_SEED = 42  #@param {type:"integer"}
SUBSET_ROOT = Path("/content/rag_dataset_subsets")
SUPPORTED_FILES = {".csv", ".tsv", ".json", ".jsonl", ".ndjson", ".parquet"}

if SUBSET_ROOT.exists():
    shutil.rmtree(SUBSET_ROOT)
SUBSET_ROOT.mkdir(parents=True, exist_ok=True)

subset_records = []

def safe_name(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", value)

def write_jsonl(rows, destination: Path) -> int:
    destination.parent.mkdir(parents=True, exist_ok=True)
    count = 0
    with destination.open("w", encoding="utf-8") as output:
        for row in rows:
            output.write(json.dumps(row, ensure_ascii=False, default=str) + "\n")
            count += 1
    return count

def record_subset(source, destination, rows, status="complete", error=""):
    subset_records.append({
        "source": str(source),
        "output": str(destination),
        "rows": rows,
        "status": status,
        "error": error,
    })

# Find Hugging Face save_to_disk roots without also processing their metadata files.
dataset_dict_roots = {path.parent for path in ROOT.rglob("dataset_dict.json")}
dataset_roots = {
    path.parent for path in ROOT.rglob("state.json")
    if not any(root == path.parent or root in path.parents for root in dataset_dict_roots)
}
hf_roots = sorted(dataset_dict_roots | dataset_roots)

for source_root in hf_roots:
    relative_root = source_root.relative_to(ROOT)
    try:
        saved = load_from_disk(str(source_root))
        splits = saved.items() if isinstance(saved, DatasetDict) else [("data", saved)]
        for split_name, split_data in splits:
            row_count = min(MAX_ROWS_PER_ITEM, len(split_data))
            sampled = split_data.shuffle(seed=SUBSET_SEED).select(range(row_count))
            destination = (
                SUBSET_ROOT / "huggingface" / relative_root / f"{safe_name(split_name)}.jsonl"
            )
            written = write_jsonl(sampled, destination)
            record_subset(f"{source_root}::{split_name}", destination, written)
            print(f"{relative_root} [{split_name}]: {written:,} rows")
    except Exception as exc:
        record_subset(source_root, "", 0, status="failed", error=repr(exc))
        print(f"Skipped Hugging Face dataset {source_root}: {exc!r}")

# Sample standalone tabular files not owned by a save_to_disk dataset.
tabular_files = sorted(
    path for path in ROOT.rglob("*")
    if path.is_file()
    and path.suffix.lower() in SUPPORTED_FILES
    and ".git" not in path.parts
    and not any(root == path.parent or root in path.parents for root in hf_roots)
)

for source_file in tabular_files:
    relative_file = source_file.relative_to(ROOT)
    destination = (
        SUBSET_ROOT / "files" / relative_file.parent / f"{relative_file.name}.sample.jsonl"
    )
    suffix = source_file.suffix.lower()
    builder = "csv" if suffix in {".csv", ".tsv"} else "json" if suffix in {".json", ".jsonl", ".ndjson"} else "parquet"
    load_kwargs = {"delimiter": "\t"} if suffix == ".tsv" else {}

    try:
        stream = load_dataset(
            builder,
            data_files=str(source_file),
            split="train",
            streaming=True,
            **load_kwargs,
        )
        stream = stream.shuffle(seed=SUBSET_SEED, buffer_size=max(10_000, MAX_ROWS_PER_ITEM))
        rows = itertools.islice(stream, MAX_ROWS_PER_ITEM)
        written = write_jsonl(rows, destination)
        record_subset(source_file, destination, written)
        print(f"{relative_file}: {written:,} rows")
    except Exception as exc:
        record_subset(source_file, destination, 0, status="failed", error=repr(exc))
        print(f"Skipped {relative_file}: {exc!r}")

subset_manifest = pd.DataFrame(subset_records)
subset_manifest_path = SUBSET_ROOT / "subset_manifest.csv"
subset_manifest.to_csv(subset_manifest_path, index=False)
display(subset_manifest)
print(f"Subset directory: {SUBSET_ROOT}")
print(f"Subset size: {human_bytes(path_size_bytes(SUBSET_ROOT))}")

## 15. Package and download the subsets

A remote Colab kernel cannot directly write to a path on your Mac. This cell creates a ZIP and displays a download link. Save or move the downloaded ZIP to `/Users/hiroakioshima/Desktop/rag_app/data/raw/`, then extract it there. If the link is unavailable in VS Code, mount the Colab server to the VS Code workspace and copy `/content/rag_dataset_subsets.zip` into that local folder.

In [34]:
from IPython.display import FileLink, display

LOCAL_DESTINATION = Path("/Users/hiroakioshima/Desktop/rag_app/data/raw")
archive_file = Path(shutil.make_archive(
    str(SUBSET_ROOT),
    "zip",
    root_dir=str(SUBSET_ROOT.parent),
    base_dir=SUBSET_ROOT.name,
))

print(f"Created {archive_file} ({human_bytes(archive_file.stat().st_size)})")
print(f"Download it, then save or move it to: {LOCAL_DESTINATION}")
display(FileLink(str(archive_file), result_html_prefix="Download subsets: "))
print("VS Code fallback: Colab: Mount Server To Workspace, then copy the ZIP from /content.")

Created /content/rag_dataset_subsets.zip (1.03 GiB)
Download it, then save or move it to: /Users/hiroakioshima/Desktop/rag_app/data/raw


/content/rag_dataset_subsets.zip

VS Code fallback: Colab: Mount Server To Workspace, then copy the ZIP from /content.


## Optional: archive the collected datasets

The following cell creates one compressed archive. It is disabled by default because archive creation temporarily requires additional disk space.

In [35]:
CREATE_ARCHIVE = False  #@param {type:"boolean"}

if CREATE_ARCHIVE:
    archive_base = Path("/content/rag_datasets")
    archive_path = shutil.make_archive(
        str(archive_base),
        "gztar",
        root_dir=str(ROOT.parent),
        base_dir=ROOT.name,
    )
    archive_size = Path(archive_path).stat().st_size
    print(f"Archive: {archive_path}")
    print(f"Archive size: {human_bytes(archive_size)} ({archive_size:,} bytes)")
else:
    print("Archive creation is disabled.")

Archive creation is disabled.


## Save the sampled datasets to Google Drive

Run the Google Drive mount cell first. This cell copies the portable subset ZIP to `MyDrive/rag_data/` and creates that directory when needed.

In [36]:
DRIVE_MYDRIVE = Path("/content/drive/MyDrive")
DRIVE_DESTINATION = DRIVE_MYDRIVE / "rag_data"

if not DRIVE_MYDRIVE.is_dir():
    raise RuntimeError(
        "Google Drive is not mounted. Run your drive.mount('/content/drive') cell first."
    )

if not SUBSET_ROOT.is_dir():
    raise RuntimeError(
        f"Subset directory not found: {SUBSET_ROOT}. Run the subset-creation cell first."
    )

source_archive = Path("/content/rag_dataset_subsets.zip")
if not source_archive.is_file():
    source_archive = Path(shutil.make_archive(
        str(SUBSET_ROOT),
        "zip",
        root_dir=str(SUBSET_ROOT.parent),
        base_dir=SUBSET_ROOT.name,
    ))

DRIVE_DESTINATION.mkdir(parents=True, exist_ok=True)
drive_archive = DRIVE_DESTINATION / source_archive.name
shutil.copy2(source_archive, drive_archive)

print(f"Saved to Google Drive: {drive_archive}")
print(f"Archive size: {human_bytes(drive_archive.stat().st_size)}")

Saved to Google Drive: /content/drive/MyDrive/rag_data/rag_dataset_subsets.zip
Archive size: 1.03 GiB
